In [1]:
import marimo as mo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from itertools import combinations
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

import threshold as th

In [ ]:
df_train = pd.read_csv("./data/train.csv", index_col="ID")
df_test = pd.read_csv("./data/test.csv", index_col="ID")

X = df_train.drop(columns="Y")
y = df_train["Y"]

cat_cols = ["x4", "x5", "x10"]

display(df_train.head())
display(df_test.head())

## Model overview

Model type: Logistic regression

Categorical predictors: `x4`, `x5`, and `x10` are one-hot encoded using `pd.get_dummies`.

Feature selection: Perform best subset selection to evaluate every subset of the available predictors. For each subset, calculate the mean F1 score from 5-fold cross-validation. Select the subset with the highest mean F1 score.

*Use `class_weight="balanced"` in logistic regression to account for class imbalance.

In [ ]:
best_score, best_cols = -1, None

for k in range(1, X.shape[1] + 1):
    for cols in combinations(X.columns, k):
        cols = list(cols)

        X_sub = pd.get_dummies(
            X[cols],
            columns=[c for c in cols if c in cat_cols]
        )

        model = LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        )

        score = cross_val_score(
            model,
            X_sub,
            y,
            cv=5,
            scoring="f1"
        ).mean()

        if score > best_score:
            best_score, best_cols = score, cols

print("Best feature subset:", best_cols)
print("Best 5-fold CV F1 score:", best_score)

## Prepare the selected features and fit the model

Combine the training and test predictors before one-hot encoding. This ensures both datasets have the same dummy-variable columns, even if a category appears in only one dataset.

In [ ]:
both = pd.concat([X[best_cols], df_test[best_cols]])

both = pd.get_dummies(
    both,
    columns=[c for c in best_cols if c in cat_cols]
)

X_best = both.iloc[:len(X)]
test_best = both.iloc[len(X):]

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_best, y)

# Keep the predicted probability of the positive class for threshold assessment.
prediction_probs = pd.Series(
    model.predict_proba(test_best)[:, 1],
    index=df_test.index,
    name="probability"
)

display(prediction_probs.head())

## Model assessment

Estimate how many test observations should be classified as positive, then visualize the estimated F-score distribution against the reported score of **0.51195**.

In [ ]:
pos_prediction_count, threshold = th.estimate_optimal_threshold(
    prediction_probs
)

display(pos_prediction_count)
display(threshold)

In [ ]:
rng = np.random.default_rng(1)

_ = th.plot_f_scores(
    empiricals=th.sample_probable_f_scores(
        rng,
        1000,
        prediction_probs,
        pos_prediction_count
    ),
    calculated=2 * threshold,
    reported=0.51195,
)

## Make the submission

Rank test observations by their predicted probability. Assign class `1` to the `pos_prediction_count` highest-probability observations and class `0` to the rest.

In [ ]:
predictions = prediction_probs.sort_values(ascending=False)

predictions.iloc[:pos_prediction_count] = 1
predictions.iloc[pos_prediction_count:] = 0

predictions = predictions.astype(np.int8).sort_index()
predictions.name = "Y"

display(predictions.head())
display(predictions.value_counts().sort_index())

In [ ]:
import pathlib

submission_filepath = pathlib.Path("./submissions/logreg.csv")
submission_filepath.parent.mkdir(parents=True, exist_ok=True)

print(f"Saving submission file to {submission_filepath}")
predictions.to_csv(submission_filepath)